# SARIMAX による時系列予測

`statsmodels` の SARIMAX モデルを使用して時系列予測を行います。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.extract import extract_data
from src.transfform import transform_data

plt.style.use("seaborn-v0_8-whitegrid")

raw_path = Path("data/raw/data.csv")
df = transform_data(extract_data(raw_path))

series = (
    df.set_index("date")["temp_avg"]
    .asfreq("D")
    .astype(float)
    .interpolate(method="time")
)

train_size = len(series) - 30
train = series.iloc[:train_size]
test = series.iloc[train_size:]

print(f"train shape: {train.shape}, test shape: {test.shape}")


In [ ]:
# SARIMAX モデルの構築と学習
# 今回は日次データであり、強力な年次季節性(365日)がありますが、
# SARIMAで周期365を設定すると計算量が膨大になるため、
# フーリエ特徴量を外生変数（exog）として与えるアプローチをとります。

def make_fourier_features(index, order=2):
    day_of_year = index.dayofyear.to_numpy()
    features = {}
    for i in range(1, order + 1):
        features[f"sin{i}"] = np.sin(2 * i * np.pi * day_of_year / 365.25)
        features[f"cos{i}"] = np.cos(2 * i * np.pi * day_of_year / 365.25)
    return pd.DataFrame(features, index=index)

exog_train = make_fourier_features(train.index, order=2)
exog_test = make_fourier_features(test.index, order=2)

# モデルの定義 (ARIMAのパラメータは簡易的に (1, 0, 1) としています)
# exog に波のデータ（フーリエ特徴量）を渡しています
sarimax_model = sm.tsa.SARIMAX(
    endog=train,
    exog=exog_train,
    order=(1, 0, 1),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_result = sarimax_model.fit(disp=False)
print(sarimax_result.summary())


In [ ]:
# 予測と評価
sarimax_forecast = sarimax_result.predict(
    start=test.index[0],
    end=test.index[-1],
    exog=exog_test
)

def score(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

sarimax_mae, sarimax_rmse = score(test, sarimax_forecast)
print(f"SARIMAX -> MAE={sarimax_mae:.3f}, RMSE={sarimax_rmse:.3f}")

# プロット
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test, label="actual", color="black", linewidth=2)
ax.plot(test.index, sarimax_forecast, label="SARIMAX", color="seagreen")
ax.set_title("30-day forecast comparison (SARIMAX)")
ax.set_ylabel("temp_avg")
ax.legend()
plt.tight_layout()
plt.show()
